In [11]:
# ── CELL 1: Imports ───────────────────────────────────────────────────────────

import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

print("Imports loaded successfully")


Imports loaded successfully


In [12]:
# ── CELL 2: Paths ────────────────────────────────────────────────────────────

QUARTERS = {
    'Q1': '/Users/mymacbook/Desktop/Booth Spring 2026/Data Visualization/Final Project/FAERS Data/Untouched/faers_ascii_2025q1/ASCII',
    'Q2': '/Users/mymacbook/Desktop/Booth Spring 2026/Data Visualization/Final Project/FAERS Data/Untouched/faers_ascii_2025q2/ASCII',
    'Q3': '/Users/mymacbook/Desktop/Booth Spring 2026/Data Visualization/Final Project/FAERS Data/Untouched/faers_ascii_2025q3/ASCII',
    'Q4': '/Users/mymacbook/Desktop/Booth Spring 2026/Data Visualization/Final Project/FAERS Data/Untouched/faers_ascii_2025Q4 (1)/ASCII',
}

OUT_DIR = '/Users/mymacbook/Desktop/Booth Spring 2026/Data Visualization/Final Project/FAERS Data/output_clean'
os.makedirs(OUT_DIR, exist_ok=True)

print("Paths configured")
for q, path in QUARTERS.items():
    print(f"  {q}: {'OK' if os.path.exists(path) else 'NOT FOUND'}")


Paths configured
  Q1: OK
  Q2: OK
  Q3: OK
  Q4: OK


In [13]:
# ── CELL 3: Load all quarters ─────────────────────────────────────────────────

FILE_TYPES = ['DEMO', 'DRUG', 'REAC', 'OUTC', 'INDI', 'RPSR', 'THER']
QUARTER_SUFFIX = {'Q1': '25Q1', 'Q2': '25Q2', 'Q3': '25Q3', 'Q4': '25Q4'}

raw = {}

for ftype in FILE_TYPES:
    frames = []
    for q, path in QUARTERS.items():
        filename = f"{ftype}{QUARTER_SUFFIX[q]}.txt"
        filepath = os.path.join(path, filename)
        if os.path.exists(filepath):
            df = pd.read_csv(filepath, sep='$', encoding='latin1', low_memory=False)
            df['quarter'] = q
            frames.append(df)
            print(f"  Loaded {filename}: {len(df):,} rows")
        else:
            print(f"  SKIPPED (not found): {filename}")
    raw[ftype] = pd.concat(frames, ignore_index=True)
    print(f"  --> {ftype} total: {len(raw[ftype]):,} rows\n")

print("All files loaded!")


  Loaded DEMO25Q1.txt: 400,514 rows
  Loaded DEMO25Q2.txt: 393,130 rows
  Loaded DEMO25Q3.txt: 438,512 rows
  Loaded DEMO25Q4.txt: 385,288 rows
  --> DEMO total: 1,617,444 rows

  Loaded DRUG25Q1.txt: 2,008,162 rows
  Loaded DRUG25Q2.txt: 1,829,056 rows
  Loaded DRUG25Q3.txt: 2,148,451 rows
  Loaded DRUG25Q4.txt: 1,815,349 rows
  --> DRUG total: 7,801,018 rows

  Loaded REAC25Q1.txt: 1,432,926 rows
  Loaded REAC25Q2.txt: 1,340,666 rows
  Loaded REAC25Q3.txt: 1,535,133 rows
  Loaded REAC25Q4.txt: 1,349,105 rows
  --> REAC total: 5,657,830 rows

  Loaded OUTC25Q1.txt: 304,027 rows
  Loaded OUTC25Q2.txt: 295,583 rows
  Loaded OUTC25Q3.txt: 343,251 rows
  Loaded OUTC25Q4.txt: 289,721 rows
  --> OUTC total: 1,232,582 rows

  Loaded INDI25Q1.txt: 1,205,861 rows
  Loaded INDI25Q2.txt: 1,136,665 rows
  Loaded INDI25Q3.txt: 1,310,274 rows
  Loaded INDI25Q4.txt: 1,168,789 rows
  --> INDI total: 4,821,589 rows

  Loaded RPSR25Q1.txt: 11,033 rows
  Loaded RPSR25Q2.txt: 11,105 rows
  Loaded RPSR25Q

In [14]:
# ── CELL 4: Deduplicate ───────────────────────────────────────────────────────

demo_raw = raw['DEMO'].copy()
demo_raw['primaryid'] = pd.to_numeric(demo_raw['primaryid'], errors='coerce')
demo_raw = demo_raw.sort_values('primaryid', ascending=False)
demo_dedup = demo_raw.drop_duplicates(subset='caseid', keep='first')

print(f"DEMO before dedup: {len(demo_raw):,}")
print(f"DEMO after dedup:  {len(demo_dedup):,}")
print(f"Duplicate cases removed: {len(demo_raw) - len(demo_dedup):,}")

valid_ids = set(demo_dedup['primaryid'])
print(f"\nValid primaryids: {len(valid_ids):,}")

for ftype in ['DRUG', 'REAC', 'OUTC', 'INDI', 'RPSR', 'THER']:
    before = len(raw[ftype])
    raw[ftype]['primaryid'] = pd.to_numeric(raw[ftype]['primaryid'], errors='coerce')
    raw[ftype] = raw[ftype][raw[ftype]['primaryid'].isin(valid_ids)]
    print(f"  {ftype}: {before:,} → {len(raw[ftype]):,} rows")



DEMO before dedup: 1,617,444
DEMO after dedup:  1,469,305
Duplicate cases removed: 148,139

Valid primaryids: 1,469,305
  DRUG: 7,801,018 → 6,645,494 rows
  REAC: 5,657,830 → 4,857,921 rows
  OUTC: 1,232,582 → 1,099,691 rows
  INDI: 4,821,589 → 4,257,461 rows
  RPSR: 43,939 → 43,938 rows
  THER: 2,002,380 → 1,720,004 rows


In [15]:
# ── CELL 5: Clean DEMO ───────────────────────────────────────────────────────

sex_map = {'M': 'Male', 'F': 'Female', 'UNK': 'Unknown'}
age_grp_map = {
    'N': 'Neonate', 'I': 'Infant', 'C': 'Child',
    'T': 'Adolescent', 'A': 'Adult', 'E': 'Elderly'
}
occp_map = {
    'MD': 'Physician', 'PH': 'Pharmacist', 'OT': 'Other HCP',
    'LW': 'Lawyer', 'CN': 'Consumer', 'HP': 'Health Professional'
}

demo = demo_dedup.copy()
demo['sex_label']     = demo['sex'].map(sex_map).fillna('Unknown')
demo['age_grp_label'] = demo['age_grp'].map(age_grp_map).fillna('Unknown')
demo['reporter_type'] = demo['occp_cod'].map(occp_map).fillna('Unknown')

demo_clean = demo[[
    'primaryid', 'caseid', 'sex_label', 'age_grp_label',
    'reporter_type', 'reporter_country', 'fda_dt', 'quarter'
]].copy()

print(f"demo_clean: {len(demo_clean):,} rows")
print("\nSex breakdown:")
print(demo_clean['sex_label'].value_counts())
print("\nAge group breakdown:")
print(demo_clean['age_grp_label'].value_counts())
print("\nReporter type breakdown:")
print(demo_clean['reporter_type'].value_counts())

demo_clean: 1,469,305 rows

Sex breakdown:
sex_label
Female     712594
Male       479311
Unknown    277400
Name: count, dtype: int64

Age group breakdown:
age_grp_label
Unknown       943001
Adult         289200
Elderly       193547
Child          21859
Adolescent     15315
Infant          4116
Neonate         2267
Name: count, dtype: int64

Reporter type breakdown:
reporter_type
Consumer               568665
Health Professional    328744
Physician              296724
Unknown                191277
Pharmacist              75074
Lawyer                   8821
Name: count, dtype: int64


In [16]:
# ── CELL 6: Clean OUTC (worst outcome per case) ───────────────────────────────

outcome_map = {
    'DE': 'Death',
    'LT': 'Life-Threatening',
    'HO': 'Hospitalization',
    'DS': 'Disability',
    'CA': 'Congenital Anomaly',
    'RI': 'Required Intervention',
    'OT': 'Other Serious'
}
severity_rank = {
    'Death': 1, 'Life-Threatening': 2, 'Hospitalization': 3,
    'Disability': 4, 'Required Intervention': 5,
    'Congenital Anomaly': 6, 'Other Serious': 7
}

outc = raw['OUTC'].copy()
outc['outcome_label'] = outc['outc_cod'].map(outcome_map).fillna('Unknown')
outc['severity_rank'] = outc['outcome_label'].map(severity_rank).fillna(99)
worst_outc = (
    outc.sort_values('severity_rank')
    .groupby('primaryid', as_index=False)
    .first()[['primaryid', 'outcome_label']]
    .rename(columns={'outcome_label': 'worst_outcome'})
)

print(f"worst_outc: {len(worst_outc):,} rows")
print("\nWorst outcome distribution:")
print(worst_outc['worst_outcome'].value_counts())


worst_outc: 823,213 rows

Worst outcome distribution:
worst_outcome
Other Serious            404626
Hospitalization          250487
Death                    110533
Life-Threatening          40695
Disability                12349
Congenital Anomaly         2332
Required Intervention      2191
Name: count, dtype: int64


In [17]:
# ── CELL 7: Clean DRUG (primary suspect only + GLP-1 flag) ───────────────────

GLP1_BRANDS   = ['OZEMPIC', 'WEGOVY', 'MOUNJARO', 'ZEPBOUND', 'RYBELSUS',
                  'VICTOZA', 'TRULICITY', 'SAXENDA']
GLP1_GENERICS = ['SEMAGLUTIDE', 'TIRZEPATIDE', 'LIRAGLUTIDE',
                  'DULAGLUTIDE', 'EXENATIDE']
GLP1_ALL = set(GLP1_BRANDS + GLP1_GENERICS)

drug = raw['DRUG'].copy()
drug['drugname'] = drug['drugname'].str.upper().str.strip()

# Primary suspect only
ps_drug = drug[drug['role_cod'] == 'PS'].copy()
ps_drug['is_glp1'] = ps_drug['drugname'].isin(GLP1_ALL)

print(f"All drug rows:            {len(drug):,}")
print(f"Primary suspect rows:     {len(ps_drug):,}")
print(f"GLP-1 primary suspect:    {ps_drug['is_glp1'].sum():,}")
print("\nGLP-1 drug breakdown:")
print(ps_drug[ps_drug['is_glp1']]['drugname'].value_counts())

All drug rows:            6,645,494
Primary suspect rows:     1,593,446
GLP-1 primary suspect:    86,839

GLP-1 drug breakdown:
drugname
ZEPBOUND       30098
MOUNJARO       27901
OZEMPIC        14766
WEGOVY          5276
TRULICITY       4339
RYBELSUS        1122
TIRZEPATIDE      960
SEMAGLUTIDE      891
VICTOZA          690
SAXENDA          541
LIRAGLUTIDE      186
DULAGLUTIDE       42
EXENATIDE         27
Name: count, dtype: int64


In [18]:
# ── CELL 8: Clean REAC ───────────────────────────────────────────────────────

DOSING_ERROR_TERMS = [
    'Incorrect dose administered',
    'Extra dose administered',
    'Accidental underdose',
    'Product dose omission issue',
    'Overdose',
    'Underdose'
]

reac = raw['REAC'].copy()
reac['pt'] = reac['pt'].str.strip()
reac['is_dosing_error'] = reac['pt'].isin(DOSING_ERROR_TERMS)
reac['is_offlabel'] = reac['pt'] == 'Off label use'

print(f"reac rows: {len(reac):,}")
print(f"Dosing error reactions: {reac['is_dosing_error'].sum():,}")
print(f"Off-label reactions:    {reac['is_offlabel'].sum():,}")
print("\nTop 15 reactions:")
print(reac['pt'].value_counts().head(15))

reac rows: 4,857,921
Dosing error reactions: 110,743
Off-label reactions:    107,825

Top 15 reactions:
pt
Off label use                  107825
Drug ineffective                85249
Fatigue                         61476
Product dose omission issue     56094
Nausea                          55341
Death                           54741
Diarrhoea                       52749
Headache                        42828
Pruritus                        40558
Dyspnoea                        39884
Pain                            38627
Arthralgia                      36719
Rash                            36584
Vomiting                        35695
Condition aggravated            34658
Name: count, dtype: int64


In [19]:
# ── CELL 9: Clean INDI ───────────────────────────────────────────────────────

indi = raw['INDI'].copy()
indi['indi_pt'] = indi['indi_pt'].str.strip()

print(f"indi rows: {len(indi):,}")
print("\nTop 15 indications:")
print(indi['indi_pt'].value_counts().head(15))


indi rows: 4,257,461

Top 15 indications:
indi_pt
Product used for unknown indication    1690251
Rheumatoid arthritis                    215585
Dermatitis atopic                        68508
Type 2 diabetes mellitus                 56950
Weight control                           56423
Asthma                                   50740
Plasma cell myeloma                      45657
Crohn's disease                          43749
Hypertension                             41543
Psoriasis                                35646
Colitis ulcerative                       32698
Diabetes mellitus                        30919
Migraine                                 29267
Prophylaxis                              29247
Ill-defined disorder                     28784
Name: count, dtype: int64


In [20]:
# ── CELL 10: Clean RPSR ──────────────────────────────────────────────────────

rpsr_map = {
    'FGN': 'Foreign',
    'SDN': 'Study',
    'LIT': 'Literature',
    'CSM': 'Consumer/Spontaneous',
    'HP':  'Health Professional',
    'UF':  'User Facility',
    'OTC': 'Over the Counter',
    'MFR': 'Manufacturer',
    'DT':  'Distributor'
}

rpsr = raw['RPSR'].copy()
rpsr['report_source'] = rpsr['rpsr_cod'].map(rpsr_map).fillna('Other')

print(f"rpsr rows: {len(rpsr):,}")
print("\nReport source breakdown:")
print(rpsr['report_source'].value_counts())


rpsr rows: 43,938

Report source breakdown:
report_source
Consumer/Spontaneous    24309
Health Professional     19160
Foreign                   469
Name: count, dtype: int64


In [21]:
# ── CELL 11: THER — therapy duration ─────────────────────────────────────────

ther = raw['THER'].copy()
ther['start_dt'] = pd.to_datetime(ther['start_dt'], format='%Y%m%d', errors='coerce')
ther['end_dt']   = pd.to_datetime(ther['end_dt'],   format='%Y%m%d', errors='coerce')
ther['duration_days'] = (ther['end_dt'] - ther['start_dt']).dt.days

valid_dur = ther['duration_days'].notna() & (ther['duration_days'] >= 0)
print(f"THER rows:               {len(ther):,}")
print(f"Rows with valid duration: {valid_dur.sum():,} ({valid_dur.mean()*100:.1f}%)")
print(f"\nDuration stats (days):")
print(ther.loc[valid_dur, 'duration_days'].describe())

THER rows:               1,720,004
Rows with valid duration: 510,880 (29.7%)

Duration stats (days):
count    510880.000000
mean        156.521140
std         621.357166
min           0.000000
25%           0.000000
50%           5.000000
75%          53.000000
max       37083.000000
Name: duration_days, dtype: float64


In [22]:
# ── CELL 12: Build master table ───────────────────────────────────────────────

master = (
    demo_clean
    .merge(worst_outc, on='primaryid', how='left')
)
master['worst_outcome'] = master['worst_outcome'].fillna('No Outcome Reported')

print(f"Master table: {len(master):,} rows")
print("\nOutcome distribution:")
print(master['worst_outcome'].value_counts())


Master table: 1,469,305 rows

Outcome distribution:
worst_outcome
No Outcome Reported      646092
Other Serious            404626
Hospitalization          250487
Death                    110533
Life-Threatening          40695
Disability                12349
Congenital Anomaly         2332
Required Intervention      2191
Name: count, dtype: int64


In [23]:
# ── CELL 13: OUTPUT — Demographics (Dashboard 1 + 2) ─────────────────────────

master.to_csv(f'{OUT_DIR}/faers_demographics.csv', index=False)
print(f"Saved faers_demographics.csv ({len(master):,} rows)")


Saved faers_demographics.csv (1,469,305 rows)


In [24]:
# ── CELL 14: OUTPUT — Drug summary (Dashboard 1 + 2 scatter) ─────────────────

drug_outc = ps_drug[['primaryid', 'drugname']].merge(worst_outc, on='primaryid', how='left')
drug_outc['worst_outcome'] = drug_outc['worst_outcome'].fillna('No Outcome Reported')

drug_summary = (
    drug_outc
    .groupby(['drugname', 'worst_outcome'])
    .size()
    .reset_index(name='report_count')
)

drug_pivot = drug_summary.pivot_table(
    index='drugname', columns='worst_outcome',
    values='report_count', fill_value=0
).reset_index()
drug_pivot.columns.name = None
drug_pivot['total_reports'] = drug_pivot.drop('drugname', axis=1).sum(axis=1)
drug_pivot['death_count']   = drug_pivot.get('Death', 0)
drug_pivot['death_rate_pct'] = (drug_pivot['death_count'] / drug_pivot['total_reports'] * 100).round(2)
drug_pivot = drug_pivot.sort_values('total_reports', ascending=False)

# Top 300 for Tableau
drug_pivot.head(300).to_csv(f'{OUT_DIR}/faers_drug_summary.csv', index=False)
print(f"Saved faers_drug_summary.csv (top 300 drugs)")
print(drug_pivot.head(10)[['drugname', 'total_reports', 'death_count', 'death_rate_pct']])


Saved faers_drug_summary.csv (top 300 drugs)
         drugname  total_reports  death_count  death_rate_pct
2103     DUPIXENT       125677.0        475.0            0.38
6294  VEDOLIZUMAB        48278.0        624.0            1.29
6616     ZEPBOUND        30098.0         51.0            0.17
4067     MOUNJARO        27901.0        278.0            1.00
5524      SKYRIZI        19963.0        725.0            3.63
5249       RINVOQ        15046.0        618.0            4.11
4557      ORGOVYX        14813.0        422.0            2.85
4609      OZEMPIC        14766.0        195.0            1.32
5202     REVLIMID        13432.0        751.0            5.59
3007       HUMIRA        11402.0        726.0            6.37


In [25]:
# ── CELL 15: OUTPUT — Drug long format (Dashboard 1) ─────────────────────────

outcome_cols = [c for c in drug_pivot.columns if c not in
                ['drugname', 'total_reports', 'death_count', 'death_rate_pct']]

drug_long = drug_pivot.head(300).melt(
    id_vars=['drugname', 'total_reports', 'death_rate_pct'],
    value_vars=outcome_cols,
    var_name='outcome',
    value_name='report_count'
)
drug_long = drug_long[drug_long['report_count'] > 0]
drug_long.to_csv(f'{OUT_DIR}/faers_drug_long.csv', index=False)
print(f"Saved faers_drug_long.csv ({len(drug_long):,} rows)")

Saved faers_drug_long.csv (2,109 rows)


In [26]:
# ── CELL 16: OUTPUT — Reaction summary (Dashboard 1) ─────────────────────────

reac_outc = reac[['primaryid', 'pt']].merge(worst_outc, on='primaryid', how='left')
reac_outc['worst_outcome'] = reac_outc['worst_outcome'].fillna('No Outcome Reported')

reac_pivot = (
    reac_outc
    .groupby(['pt', 'worst_outcome'])
    .size()
    .reset_index(name='count')
    .pivot_table(index='pt', columns='worst_outcome', values='count', fill_value=0)
    .reset_index()
)
reac_pivot.columns.name = None
reac_pivot['total'] = reac_pivot.drop('pt', axis=1).sum(axis=1)
reac_pivot = reac_pivot.sort_values('total', ascending=False)
reac_pivot.head(150).to_csv(f'{OUT_DIR}/faers_reaction_summary.csv', index=False)
print(f"Saved faers_reaction_summary.csv (top 150 reactions)")
print(reac_pivot.head(10)[['pt', 'total']])

Saved faers_reaction_summary.csv (top 150 reactions)
                                pt     total
10883                Off label use  107825.0
4803              Drug ineffective   85249.0
5796                       Fatigue   61476.0
12610  Product dose omission issue   56094.0
10331                       Nausea   55341.0
4272                         Death   54741.0
4604                     Diarrhoea   52749.0
6913                      Headache   42828.0
12803                     Pruritus   40558.0
4932                      Dyspnoea   39884.0


In [27]:
# ── CELL 17: OUTPUT — KPIs (Dashboard 1) ─────────────────────────────────────

kpis = pd.DataFrame([{
    'total_reports':      len(master),
    'total_deaths':       (master['worst_outcome'] == 'Death').sum(),
    'total_hospitalized': (master['worst_outcome'] == 'Hospitalization').sum(),
    'unique_drugs':       ps_drug['drugname'].nunique(),
    'unique_reactions':   reac['pt'].nunique(),
    'glp1_reports':       ps_drug['is_glp1'].sum(),
    'offlabel_reports':   reac['is_offlabel'].sum(),
}])

kpis.to_csv(f'{OUT_DIR}/faers_kpis.csv', index=False)
print("Saved faers_kpis.csv")
print(kpis.T)


Saved faers_kpis.csv
                          0
total_reports       1469305
total_deaths         110533
total_hospitalized   250487
unique_drugs           6720
unique_reactions      16762
glp1_reports          86839
offlabel_reports     107825


In [28]:
# ── CELL 18: OUTPUT — Reporter x Outcome (Dashboard 2, H5) ───────────────────

reporter_outc = (
    master
    .groupby(['reporter_type', 'worst_outcome'])
    .size()
    .reset_index(name='count')
)
reporter_outc.to_csv(f'{OUT_DIR}/faers_reporter_outcome.csv', index=False)
print(f"Saved faers_reporter_outcome.csv ({len(reporter_outc):,} rows)")
print(reporter_outc.pivot_table(index='reporter_type', columns='worst_outcome',
                                 values='count', fill_value=0))

Saved faers_reporter_outcome.csv (48 rows)
worst_outcome        Congenital Anomaly    Death  Disability  Hospitalization  \
reporter_type                                                                   
Consumer                          195.0  24987.0      6260.0          69697.0   
Health Professional               790.0  27020.0      1236.0          53018.0   
Lawyer                              2.0    370.0       651.0           1138.0   
Pharmacist                         63.0   6538.0       676.0          20333.0   
Physician                        1082.0  33127.0      2437.0          66563.0   
Unknown                           200.0  18491.0      1089.0          39738.0   

worst_outcome        Life-Threatening  No Outcome Reported  Other Serious  \
reporter_type                                                               
Consumer                       5338.0             333297.0       128134.0   
Health Professional           10687.0             143317.0        92473.0   


In [29]:
# ── CELL 19: OUTPUT — Age x Outcome (Dashboard 2, H4) ────────────────────────

age_outc = (
    master
    .groupby(['age_grp_label', 'worst_outcome'])
    .size()
    .reset_index(name='count')
)
age_outc.to_csv(f'{OUT_DIR}/faers_age_outcome.csv', index=False)
print(f"Saved faers_age_outcome.csv ({len(age_outc):,} rows)")

# Show missingness so we know if it's usable
unknown_pct = (master['age_grp_label'] == 'Unknown').mean() * 100
print(f"\nAge group missingness: {unknown_pct:.1f}% Unknown")

Saved faers_age_outcome.csv (53 rows)

Age group missingness: 64.2% Unknown


In [30]:
# ── CELL 20: OUTPUT — Off-label comparison (Dashboard 2, H6) ─────────────────

offlabel_ids = set(reac[reac['is_offlabel']]['primaryid'])
master['offlabel_flag'] = master['primaryid'].isin(offlabel_ids)
master['offlabel_flag'] = master['offlabel_flag'].map({True: 'Off-Label', False: 'On-Label'})

offlabel_outc = (
    master
    .groupby(['offlabel_flag', 'worst_outcome'])
    .size()
    .reset_index(name='count')
)

# Add percentage within each group
totals = master['offlabel_flag'].value_counts().rename('group_total')
offlabel_outc = offlabel_outc.merge(totals, left_on='offlabel_flag', right_index=True)
offlabel_outc['pct'] = (offlabel_outc['count'] / offlabel_outc['group_total'] * 100).round(2)

offlabel_outc.to_csv(f'{OUT_DIR}/faers_offlabel_comparison.csv', index=False)
print(f"Saved faers_offlabel_comparison.csv")
print(offlabel_outc.pivot_table(index='offlabel_flag', columns='worst_outcome',
                                 values='pct', fill_value=0).round(1))


Saved faers_offlabel_comparison.csv
worst_outcome  Congenital Anomaly  Death  Disability  Hospitalization  \
offlabel_flag                                                           
Off-Label                     0.0    7.4         0.6             17.2   
On-Label                      0.2    7.5         0.9             17.0   

worst_outcome  Life-Threatening  No Outcome Reported  Other Serious  \
offlabel_flag                                                         
Off-Label                   2.8                 39.3           32.6   
On-Label                    2.8                 44.3           27.2   

worst_outcome  Required Intervention  
offlabel_flag                         
Off-Label                        0.0  
On-Label                         0.2  


In [31]:
# ── CELL 21: OUTPUT — GLP-1 outcomes (Dashboard 3) ───────────────────────────

glp1_ps = ps_drug[ps_drug['is_glp1']][['primaryid', 'drugname', 'route']].copy()

glp1_outc = glp1_ps.merge(worst_outc, on='primaryid', how='left')
glp1_outc['worst_outcome'] = glp1_outc['worst_outcome'].fillna('No Outcome Reported')
glp1_outc.to_csv(f'{OUT_DIR}/glp1_outcomes.csv', index=False)
print(f"Saved glp1_outcomes.csv ({len(glp1_outc):,} rows)")
print("\nGLP-1 outcomes:")
print(glp1_outc['worst_outcome'].value_counts())


Saved glp1_outcomes.csv (86,839 rows)

GLP-1 outcomes:
worst_outcome
No Outcome Reported      60751
Other Serious            14875
Hospitalization           8564
Life-Threatening          1025
Disability                 800
Death                      696
Required Intervention      100
Congenital Anomaly          28
Name: count, dtype: int64


In [32]:
# ── CELL 22: OUTPUT — GLP-1 reactions (Dashboard 3) ──────────────────────────

glp1_reac = glp1_ps.merge(reac[['primaryid', 'pt', 'is_dosing_error']], on='primaryid', how='left')
glp1_reac.to_csv(f'{OUT_DIR}/glp1_reactions.csv', index=False)
print(f"Saved glp1_reactions.csv ({len(glp1_reac):,} rows)")
print("\nTop 20 GLP-1 reactions:")
print(glp1_reac['pt'].value_counts().head(20))

Saved glp1_reactions.csv (204,828 rows)

Top 20 GLP-1 reactions:
pt
Incorrect dose administered                 12160
Nausea                                       9834
Diarrhoea                                    5911
Vomiting                                     5910
Injection site pain                          5456
Off label use                                4623
Constipation                                 4363
Extra dose administered                      4267
Drug ineffective                             3259
Impaired gastric emptying                    2961
Decreased appetite                           2786
Abdominal pain upper                         2474
Fatigue                                      2427
Weight increased                             2389
Accidental underdose                         2339
Product dose omission issue                  2225
Abdominal pain                               2128
Wrong technique in product usage process     2012
Headache                        

In [33]:
# ── CELL 23: OUTPUT — GLP-1 demographics (Dashboard 3) ───────────────────────

glp1_demo = glp1_ps.merge(demo_clean, on='primaryid', how='left')
glp1_demo = glp1_demo.merge(worst_outc, on='primaryid', how='left')
glp1_demo['worst_outcome'] = glp1_demo['worst_outcome'].fillna('No Outcome Reported')
glp1_demo.to_csv(f'{OUT_DIR}/glp1_demographics.csv', index=False)
print(f"Saved glp1_demographics.csv ({len(glp1_demo):,} rows)")

Saved glp1_demographics.csv (86,839 rows)


In [34]:
# ── CELL 24: OUTPUT — GLP-1 indications (Dashboard 3) ────────────────────────

glp1_indi = glp1_ps.merge(indi[['primaryid', 'indi_pt']], on='primaryid', how='left')
glp1_indi.to_csv(f'{OUT_DIR}/glp1_indications.csv', index=False)
print(f"Saved glp1_indications.csv ({len(glp1_indi):,} rows)")
print("\nTop 15 GLP-1 indications:")
print(glp1_indi['indi_pt'].value_counts().head(15))


Saved glp1_indications.csv (276,833 rows)

Top 15 GLP-1 indications:
indi_pt
Product used for unknown indication    122179
Weight control                          55271
Type 2 diabetes mellitus                37495
Diabetes mellitus                       13673
Glucose tolerance impaired               4179
Sleep apnoea syndrome                    3701
Hypertension                             2792
Obesity                                  2527
Blood cholesterol increased              1619
Obstructive sleep apnoea syndrome        1424
Pain                                     1108
Glycosylated haemoglobin increased       1080
Depression                               1012
Weight decreased                         1000
Anxiety                                   899
Name: count, dtype: int64


In [35]:
# ── CELL 25: OUTPUT — GLP-1 dosing errors vs all drugs (Dashboard 3, H2) ─────

# Dosing error rate for GLP-1 cases
glp1_ids = set(glp1_ps['primaryid'])
reac['is_glp1_case'] = reac['primaryid'].isin(glp1_ids)

dosing_summary = (
    reac
    .groupby('is_glp1_case')
    .agg(
        total_reactions=('primaryid', 'count'),
        dosing_errors=('is_dosing_error', 'sum')
    )
    .reset_index()
)
dosing_summary['dosing_error_pct'] = (
    dosing_summary['dosing_errors'] / dosing_summary['total_reactions'] * 100
).round(2)
dosing_summary['group'] = dosing_summary['is_glp1_case'].map(
    {True: 'GLP-1 Drugs', False: 'All Other Drugs'}
)

dosing_summary.to_csv(f'{OUT_DIR}/glp1_dosing_vs_all.csv', index=False)
print("Saved glp1_dosing_vs_all.csv")
print(dosing_summary[['group', 'total_reactions', 'dosing_errors', 'dosing_error_pct']])


Saved glp1_dosing_vs_all.csv
             group  total_reactions  dosing_errors  dosing_error_pct
0  All Other Drugs          4653129          89143              1.92
1      GLP-1 Drugs           204792          21600             10.55


In [36]:
# ── CELL 26: OUTPUT — GLP-1 therapy duration (Dashboard 3) ───────────────────

glp1_ther = glp1_ps.merge(ther[['primaryid', 'start_dt', 'end_dt', 'duration_days']],
                           on='primaryid', how='left')
glp1_ther_valid = glp1_ther[glp1_ther['duration_days'].notna() &
                              (glp1_ther['duration_days'] >= 0) &
                              (glp1_ther['duration_days'] <= 3650)]

glp1_ther_valid.to_csv(f'{OUT_DIR}/glp1_therapy_duration.csv', index=False)
print(f"Saved glp1_therapy_duration.csv ({len(glp1_ther_valid):,} valid rows)")
print(f"Coverage: {len(glp1_ther_valid)/len(glp1_ps)*100:.1f}% of GLP-1 cases have duration data")
print(glp1_ther_valid['duration_days'].describe())


Saved glp1_therapy_duration.csv (19,111 valid rows)
Coverage: 22.0% of GLP-1 cases have duration data
count    19111.000000
mean       280.582858
std        476.758985
min          0.000000
25%         23.000000
50%         83.000000
75%        336.000000
max       3622.000000
Name: duration_days, dtype: float64


In [37]:
# ── CELL 27: Final summary ────────────────────────────────────────────────────

print("=" * 50)
print("ALL OUTPUT FILES SAVED")
print("=" * 50)
for f in sorted(os.listdir(OUT_DIR)):
    fpath = os.path.join(OUT_DIR, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {f}: {size_kb:.0f} KB")


ALL OUTPUT FILES SAVED
  faers_age_outcome.csv: 1 KB
  faers_demographics.csv: 109423 KB
  faers_drug_long.csv: 90 KB
  faers_drug_summary.csv: 21 KB
  faers_kpis.csv: 0 KB
  faers_offlabel_comparison.csv: 1 KB
  faers_reaction_summary.csv: 11 KB
  faers_reporter_outcome.csv: 2 KB
  glp1_demographics.csv: 7910 KB
  glp1_dosing_vs_all.csv: 0 KB
  glp1_indications.csv: 14766 KB
  glp1_outcomes.csv: 3890 KB
  glp1_reactions.csv: 10267 KB
  glp1_therapy_duration.csv: 1007 KB


In [38]:
family_map = {
    'MOUNJARO':    'Tirzepatide', 'ZEPBOUND':    'Tirzepatide', 'TIRZEPATIDE': 'Tirzepatide',
    'OZEMPIC':     'Semaglutide', 'WEGOVY':      'Semaglutide', 'RYBELSUS':    'Semaglutide', 'SEMAGLUTIDE': 'Semaglutide',
    'VICTOZA':     'Liraglutide', 'SAXENDA':     'Liraglutide', 'LIRAGLUTIDE': 'Liraglutide',
    'TRULICITY':   'Dulaglutide', 'DULAGLUTIDE': 'Dulaglutide',
    'EXENATIDE':   'Exenatide'
}

for fname in ['glp1_outcomes.csv', 'glp1_reactions.csv', 'glp1_demographics.csv', 'glp1_indications.csv', 'glp1_therapy_duration.csv']:
    fpath = f'{OUT_DIR}/{fname}'
    df = pd.read_csv(fpath)
    df['drug_family'] = df['drugname'].map(family_map)
    df.to_csv(fpath, index=False)
    print(f"Updated {fname}")

print("\nDrug family totals:")
df_check = pd.read_csv(f'{OUT_DIR}/glp1_outcomes.csv')
print(df_check['drug_family'].value_counts())

Updated glp1_outcomes.csv
Updated glp1_reactions.csv
Updated glp1_demographics.csv
Updated glp1_indications.csv
Updated glp1_therapy_duration.csv

Drug family totals:
drug_family
Tirzepatide    58959
Semaglutide    22055
Dulaglutide     4381
Liraglutide     1417
Exenatide         27
Name: count, dtype: int64


In [39]:
reporter_outc = pd.read_csv(f'{OUT_DIR}/faers_reporter_outcome.csv')
totals = reporter_outc.groupby('reporter_type')['count'].transform('sum')
reporter_outc['pct_within_group'] = (reporter_outc['count'] / totals * 100).round(2)
reporter_outc.to_csv(f'{OUT_DIR}/faers_reporter_outcome.csv', index=False)
print("Updated faers_reporter_outcome.csv with percentages")
print(reporter_outc[reporter_outc['worst_outcome'] == 'Death'].sort_values('pct_within_group', ascending=False))

Updated faers_reporter_outcome.csv with percentages
          reporter_type worst_outcome  count  pct_within_group
33            Physician         Death  33127             11.16
41              Unknown         Death  18491              9.67
25           Pharmacist         Death   6538              8.71
9   Health Professional         Death  27020              8.22
1              Consumer         Death  24987              4.39
17               Lawyer         Death    370              4.19


In [40]:
quarterly = (
    master
    .groupby(['quarter', 'worst_outcome'])
    .size()
    .reset_index(name='count')
)
totals_q = master.groupby('quarter')['primaryid'].count().rename('quarter_total')
quarterly = quarterly.merge(totals_q, on='quarter')
quarterly['pct'] = (quarterly['count'] / quarterly['quarter_total'] * 100).round(2)
quarterly.to_csv(f'{OUT_DIR}/faers_quarterly_trend.csv', index=False)
print("Saved faers_quarterly_trend.csv")
print(quarterly.pivot_table(index='quarter', columns='worst_outcome', values='count', fill_value=0))

Saved faers_quarterly_trend.csv
worst_outcome  Congenital Anomaly    Death  Disability  Hospitalization  \
quarter                                                                   
Q1                          499.0  27336.0      2749.0          54816.0   
Q2                          735.0  23527.0      3027.0          57245.0   
Q3                          582.0  33379.0      3432.0          70603.0   
Q4                          516.0  26291.0      3141.0          67823.0   

worst_outcome  Life-Threatening  No Outcome Reported  Other Serious  \
quarter                                                               
Q1                       8473.0             152785.0        86107.0   
Q2                       9841.0             151578.0        91873.0   
Q3                      11711.0             174783.0       117173.0   
Q4                      10670.0             166946.0       109473.0   

worst_outcome  Required Intervention  
quarter                               
Q1          

In [41]:
country_outc = (
    master
    .groupby(['reporter_country', 'worst_outcome'])
    .size()
    .reset_index(name='count')
)
country_totals = master.groupby('reporter_country')['primaryid'].count().rename('total')
country_outc = country_outc.merge(country_totals, on='reporter_country')
country_outc['pct'] = (country_outc['count'] / country_outc['total'] * 100).round(2)

# Only keep countries with at least 100 reports to avoid noise
valid_countries = country_totals[country_totals >= 100].index
country_outc = country_outc[country_outc['reporter_country'].isin(valid_countries)]

country_outc.to_csv(f'{OUT_DIR}/faers_country_outcome.csv', index=False)
print(f"Saved faers_country_outcome.csv")
print(f"Countries included: {country_outc['reporter_country'].nunique()}")
print(country_outc.groupby('reporter_country')['total'].first().sort_values(ascending=False).head(15))

Saved faers_country_outcome.csv
Countries included: 81
reporter_country
US    972239
EU     87852
CA     87205
GB     56947
JP     55074
CN     33964
FR     24739
AU     12675
DE     12550
BR     10197
IN      9390
IT      8735
ES      8279
CO      7619
KR      4869
Name: total, dtype: int64
